# Battery BESS + Generateur PV — Optimisation Arbitrage Spot Price

## Vue d'ensemble

Ce notebook etend le modele d'arbitrage BESS sur le **marche spot France (EPEX)** en ajoutant un **generateur PV** connecte en priorite a la batterie.

**Sources de revenu :**
1. **Arbitrage spot** : decharge batterie quand prix eleve, charge quand prix bas
2. **Surplus PV** : vente directe du PV non stocke au tarif de rachat fixe

**Modele PV :**
- Distribution saisonniere : rayonnement extraterrestre H0(latitude, DOJ)
- Distribution intraday : gaussienne centree sur le midi solaire local
- Equation du temps pour le midi solaire precis

**Modele LP etendu :**
- Variables : charge_reseau, decharge, vente_surplus_PV
- PV alimente la batterie en priorite (LP route le PV vers batterie si profitable)
- Reseau complete la charge quand PV insuffisant

---
*Donnees : EPEX Spot France — prix spot horaires historiques (EUR/MWh), heure locale France*

In [ ]:
%matplotlib inline

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.optimize import linprog
from datetime import date, timedelta
import time
import warnings
warnings.filterwarnings('ignore')

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    _HAS_WIDGETS = True
except ImportError:
    _HAS_WIDGETS = False
    print('ipywidgets non installe. pip install ipywidgets')

try:
    from ipyleaflet import Map, Marker, basemaps
    _HAS_LEAFLET = True
except ImportError:
    _HAS_LEAFLET = False
    print('ipyleaflet non installe. pip install ipyleaflet')

plt.rcParams.update({
    'figure.figsize': (14, 5),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Libraries loaded OK')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  PARAMETRES — modifier selon votre projet
# ═══════════════════════════════════════════════════════════════════════════

# ── Batterie ────────────────────────────────────────────────────────────────
CAPACITY_KWH    = 5680.0    # Capacite nominale                    [kWh]
C_RATE          = 0.5       # C-rate : P_max = C x Capacite        [h-1]
CONNECTION_KW   = 3000.0    # Puissance raccordement Enedis         [kW]
EFF_ROUNDTRIP   = 0.97      # Rendement aller-retour                [-]
AGING_PER_FEC   = 1e-5      # Perte capacite par cycle equivalent   [-/FEC]
CAPACITY_EOL    = 0.80      # Fin de vie (80%)                      [-]

# ── Etat de charge (SOC) ────────────────────────────────────────────────────
SOC_MIN_PCT     = 0.10
SOC_MAX_PCT     = 0.90
SOC_INIT_PCT    = 0.50

# ── Agregateur ──────────────────────────────────────────────────────────────
AGGREGATOR_SPREAD        = 0.000   # [EUR/kWh]
MIN_DISCHARGE_SPREAD_MWH = 10.0    # [EUR/MWh]

# ── Generateur PV ───────────────────────────────────────────────────────────
PV_CAPACITY_KWP   = 100.0   # Capacite installee                   [kWc]
PV_SPECIFIC_YIELD = 1200.0  # Productible specifique               [kWh/kWc/an]
PV_FIT_PRICE      = 0.10    # Tarif rachat surplus PV              [EUR/kWh]

# ── Strategie de routage PV ─────────────────────────────────────────────────
#   1 = Economique    : le LP arbitre librement entre stocker le PV dans la
#       batterie et le vendre au tarif FIT selon la rentabilite globale sur
#       l'horizon de planification.
#   2 = Priorite batt.: le PV est toujours route vers la batterie en premier.
#       Il n'est vendu au FIT que si la batterie est pleine (SOC max atteint).
PV_STRATEGY = 2

# ── Localisation PV (France) ─────────────────────────────────────────────────
#    Modifier via le widget carte dans la cellule suivante,
#    ou directement ici.
PV_LATITUDE   = 46.5    # Latitude  [degres N]  (France : 41 - 51.5)
PV_LONGITUDE  = 2.3     # Longitude [degres E]  (France : -5.5 - 9.5)

# ── Chemins fichiers ────────────────────────────────────────────────────────
START_DATE      = date(2025, 1, 1)
CSV_DATA_PATH   = 'data_France_historical-spot_price_hourly.csv'
OUTPUT_CSV_PATH = 'battery_PV_optimization_results.csv'

# ── Constantes derivees ──────────────────────────────────────────────────────
ETA_C = EFF_ROUNDTRIP ** 0.5
ETA_D = EFF_ROUNDTRIP ** 0.5
P_MAX = min(C_RATE * CAPACITY_KWH, CONNECTION_KW)
MIN_DISCHARGE_SPREAD_KWH = MIN_DISCHARGE_SPREAD_MWH / 1000.0

_strat_label = {1: 'Economique (arbitrage FIT/spot)',
                2: 'Priorite batterie (PV → batt. en premier)'}

print('=' * 62)
print('  CONFIGURATION BATTERIE + PV')
print('=' * 62)
print(f'  Batterie')
print(f'    Capacite nominale     : {CAPACITY_KWH:>10,.0f} kWh')
print(f'    P_max                 : {P_MAX:>10,.0f} kW')
print(f'    Rendement RT          : {EFF_ROUNDTRIP*100:>9.1f} %')
print(f'    Plage SOC             : {SOC_MIN_PCT*100:.0f}% - {SOC_MAX_PCT*100:.0f}%')
print(f'')
print(f'  Generateur PV')
print(f'    Capacite              : {PV_CAPACITY_KWP:>10,.0f} kWc')
print(f'    Productible specifique: {PV_SPECIFIC_YIELD:>10,.0f} kWh/kWc')
print(f'    Production annuelle   : {PV_CAPACITY_KWP*PV_SPECIFIC_YIELD:>10,.0f} kWh/an')
print(f'    Tarif rachat FIT      : {PV_FIT_PRICE*100:>10.1f} c EUR/kWh')
print(f'    Strategie routage     : Strat. {PV_STRATEGY} — {_strat_label[PV_STRATEGY]}')
print(f'    Localisation          : {PV_LATITUDE:.4f} N, {PV_LONGITUDE:.4f} E')
print('=' * 62)

In [ ]:
# ── Carte interactive pour la localisation du generateur PV ─────────────────
# Glisser le marqueur orange pour positionner le PV.
# Les coordonnees sont mises a jour automatiquement dans PV_LATITUDE / PV_LONGITUDE.

if _HAS_LEAFLET and _HAS_WIDGETS:
    _map = Map(center=(PV_LATITUDE, PV_LONGITUDE), zoom=6,
               basemap=basemaps.OpenStreetMap.Mapnik)
    _map.layout.height = '400px'
    _marker = Marker(
        location=(PV_LATITUDE, PV_LONGITUDE),
        draggable=True,
        title='Generateur PV — glisser pour repositionner',
    )
    _map.add_layer(_marker)

    _coord_lbl = widgets.Label(
        value=f'Lat : {PV_LATITUDE:.4f} N  |  Lon : {PV_LONGITUDE:.4f} E')

    def _on_move(change):
        global PV_LATITUDE, PV_LONGITUDE
        PV_LATITUDE  = round(_marker.location[0], 4)
        PV_LONGITUDE = round(_marker.location[1], 4)
        _coord_lbl.value = (f'Lat : {PV_LATITUDE:.4f} N  '
                            f'|  Lon : {PV_LONGITUDE:.4f} E  '
                            f'[mise a jour]')

    _marker.observe(_on_move, names=['location'])

    display(widgets.VBox([
        widgets.HTML(
            '<b style="color:#1a3a5c">Localisation generateur PV'
            ' — Glissez le marqueur sur la carte</b>'),
        _map,
        _coord_lbl,
    ]))
else:
    print('Carte non disponible (ipyleaflet non installe).')
    print(f'Localisation courante : {PV_LATITUDE:.4f} N, {PV_LONGITUDE:.4f} E')
    print('Modifier PV_LATITUDE et PV_LONGITUDE dans la cellule parametres.')

In [ ]:
# ── Chargement donnees spot price ─────────────────────────────────────────────
df_raw = pd.read_csv(CSV_DATA_PATH, sep=';')
df_raw['datetime'] = pd.to_datetime(
    df_raw['Datetime (Local)'], format='%d.%m.%Y %H:%M')
df_raw['price_eur_kwh'] = (
    pd.to_numeric(df_raw['Spot Price (EUR/MWhe)'], errors='coerce') / 1000.0)

df = (df_raw[['datetime', 'price_eur_kwh']]
      .dropna().sort_values('datetime')
      .drop_duplicates('datetime').set_index('datetime'))

df_sim = df[df.index.date >= START_DATE].copy()

print(f'Dataset      : {df.index.min().date()} => {df.index.max().date()}  ({len(df):,} h)')
print(f'Simulation   : {df_sim.index.min().date()} => {df_sim.index.max().date()}  ({len(df_sim):,} h)')
print(f'Prix negatifs: {(df_sim.price_eur_kwh < 0).sum()} heures')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MODELE SOLAIRE — Profil horaire PV
# ══════════════════════════════════════════════════════════════════════════════

def _solar_params(doy, lat_rad):
    """Rayonnement extraterrestre normalise H0, duree du jour (h)."""
    decl   = np.radians(23.45 * np.sin(np.radians(360 / 365 * (doy - 81))))
    cos_ha = float(np.clip(-np.tan(lat_rad) * np.tan(decl), -1.0, 1.0))
    ha_rad = np.arccos(cos_ha)
    ha_deg = np.degrees(ha_rad)
    H0 = max(
        np.cos(lat_rad) * np.cos(decl) * np.sin(ha_rad) +
        ha_rad * np.sin(lat_rad) * np.sin(decl), 0.0)
    return float(H0), 2.0 * ha_deg / 15.0


def _solar_noon_local(doy, longitude, month):
    """Midi solaire en heure legale France (UTC+1 hiver / UTC+2 ete)."""
    B        = np.radians(360 / 365 * (doy - 81))
    eot_min  = 9.87 * np.sin(2*B) - 7.53 * np.cos(B) - 1.5 * np.sin(B)
    noon_utc = 12.0 - longitude / 15.0 - eot_min / 60.0
    tz = 2 if 4 <= month <= 10 else 1
    return noon_utc + tz


def compute_pv_profile(capacity_kwp, specific_yield, latitude, longitude, dt_index):
    """
    Calcule le profil horaire de production PV (kWh/h) pour chaque heure de dt_index.

    Modele :
      - Energie annuelle = capacity_kwp x specific_yield kWh
      - Distribution saisonniere : H0(latitude, DOJ) / mean(H0)
      - Distribution intraday    : Gaussienne centree sur midi solaire
        sigma = day_length / 4  (2 sigma ~ toute la periode d'ensoleillement)
    """
    annual_kwh = capacity_kwp * specific_yield
    lat_rad    = np.radians(latitude)

    # H0 pour les 365 DOJ
    H0_doy = np.zeros(366)
    for doy in range(1, 366):
        H0_doy[doy], _ = _solar_params(doy, lat_rad)
    H0_mean = H0_doy[1:366].mean()
    if H0_mean < 1e-9:
        return np.zeros(len(dt_index))

    pv    = np.zeros(len(dt_index))
    dates = np.array([ts.date() for ts in dt_index])

    for d in sorted(set(dates)):
        mask    = dates == d
        indices = np.where(mask)[0]
        hours   = np.array([dt_index[i].hour for i in indices])
        doy     = dt_index[indices[0]].timetuple().tm_yday
        month   = dt_index[indices[0]].month

        H0, day_length = _solar_params(doy, lat_rad)
        if day_length < 0.5:
            continue

        solar_noon  = _solar_noon_local(doy, longitude, month)
        sigma       = max(day_length / 4.0, 0.5)
        sunrise     = solar_noon - day_length / 2.0
        sunset      = solar_noon + day_length / 2.0
        daily_kwh   = annual_kwh / 365.0 * (H0 / H0_mean)

        weights = np.array([
            np.exp(-0.5 * ((h + 0.5 - solar_noon) / sigma) ** 2)
            if sunrise < h + 0.5 < sunset else 0.0
            for h in hours
        ])
        w_sum = weights.sum()
        if w_sum > 0:
            pv[indices] = weights / w_sum * daily_kwh

    return pv


# ── Generation du profil PV pour la periode de simulation ─────────────────────
print('Calcul du profil PV...')
t0 = time.time()
pv_array = compute_pv_profile(
    PV_CAPACITY_KWP, PV_SPECIFIC_YIELD, PV_LATITUDE, PV_LONGITUDE, df_sim.index)
pv_profile = pd.Series(pv_array, index=df_sim.index, name='pv_kwh')
print(f'OK en {time.time()-t0:.2f}s')
print(f'Production PV simulee : {pv_array.sum():,.0f} kWh'
      f'  (cible annuelle : {PV_CAPACITY_KWP*PV_SPECIFIC_YIELD:,.0f} kWh)')
print(f'Pic horaire           : {pv_array.max():.1f} kWh/h'
      f'  ({pv_array.max()/PV_CAPACITY_KWP*100:.1f}% de la puissance installee)')

# Apercu profil moyen journalier par mois
pv_df = pd.DataFrame({'pv': pv_array}, index=df_sim.index)
pv_df['month'] = pv_df.index.strftime('%Y-%m')
pv_df['hour']  = pv_df.index.hour
pv_by_month = pv_df.groupby(['month', 'hour'])['pv'].mean().unstack('month')

fig, ax = plt.subplots(figsize=(14, 4))
for col in pv_by_month.columns:
    ax.plot(pv_by_month.index, pv_by_month[col], lw=1.2, alpha=0.7, label=col)
ax.plot(pv_by_month.index, pv_by_month.mean(axis=1), 'k--', lw=2, label='Moyenne')
ax.set_xlabel('Heure locale')
ax.set_ylabel('kWh/h (moyenne journaliere)')
ax.set_title(f'Profil horaire moyen PV par mois — {PV_CAPACITY_KWP:.0f} kWc'
             f' @ {PV_LATITUDE:.2f}N, {PV_LONGITUDE:.2f}E')
ax.set_xticks(range(24))
ax.set_xticklabels([f'{h}h' for h in range(24)], rotation=45, fontsize=8)
ax.legend(fontsize=8, ncol=4, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  OPTIMISEUR LP ETENDU — avec generateur PV
# ══════════════════════════════════════════════════════════════════════════════
#
# Variables x = [ch_grid(T), di(T), pv_sell(T)]
#   ch_grid  : charge depuis reseau [kWh]
#   di       : decharge vers reseau [kWh]
#   pv_sell  : vente directe surplus PV au tarif FIT [kWh]
#   pv_to_bat = pv_kwh - pv_sell   (PV vers batterie, variable implicite)
#
# Objectif (min cout) :
#   min sum_t [ch_grid[t]*spot[t] - di[t]*resale[t] + pv_sell[t]*coeff_pv]
#
# coeff_pv selon strategie :
#   1 = Economique    : coeff = -fit_price  => LP arbitre stockage vs vente FIT
#   2 = Priorite batt.: coeff = 1000 - fit_price  => vente uniquement si SOC_max sature
#
# Contrainte puissance totale + anti-simultaneite :
#   ch_grid[t] + pv_to_bat[t] + di[t] <= P_max
#   <=>  ch_grid[t] - pv_sell[t] + di[t] <= P_max - pv[t]
#
# Contraintes SOC (2T lignes) :
#   SOC_init + eta_c*sum_{k<=t}(ch_grid[k] + pv[k] - pv_sell[k])
#            - sum_{k<=t} di[k]/eta_d  in [SOC_min, SOC_max]

def optimize_schedule_pv(prices, pv_kwh, soc_init_kwh, capacity_kwh,
                          p_max, eta_c, eta_d, soc_min_pct, soc_max_pct,
                          agg_spread, fit_price, pv_strategy=2):
    """
    pv_strategy :
      1 = Economique    — LP arbitre librement stockage vs vente FIT.
      2 = Priorite batt — Penalite large, vente uniquement si SOC_max sature.
    """
    T  = len(prices)
    pv = np.asarray(pv_kwh, dtype=float)
    if T == 0:
        return np.zeros(0), np.zeros(0), np.zeros(0), np.zeros(0)

    resale      = prices + agg_spread
    soc_min_kwh = soc_min_pct * capacity_kwh
    soc_max_kwh = soc_max_pct * capacity_kwh

    if pv_strategy == 1:
        _pv_sell_cost = -fit_price           # revenue FIT => LP arbitre vs spot
    else:
        _pv_sell_cost = 1000.0 - fit_price   # penalite >> tout prix realisable
    c_obj     = np.concatenate([prices, -resale, _pv_sell_cost * np.ones(T)])
    cumsum_pv = np.cumsum(eta_c * pv)

    A_rows, b_rows = [], []

    for t in range(T):
        row = np.zeros(3 * T)
        row[:t + 1]          =  eta_c
        row[T:T + t + 1]     = -1.0 / eta_d
        row[2*T:2*T + t + 1] = -eta_c
        A_rows.append(row.copy());  b_rows.append(soc_max_kwh - soc_init_kwh - cumsum_pv[t])
        A_rows.append(-row.copy()); b_rows.append(soc_init_kwh - soc_min_kwh + cumsum_pv[t])

    for t in range(T):
        row = np.zeros(3 * T)
        row[t]       =  1.0
        row[T + t]   =  1.0
        row[2*T + t] = -1.0
        A_rows.append(row)
        b_rows.append(p_max - pv[t])

    bounds = (
        [(0.0, p_max)] * T +
        [(0.0, p_max)] * T +
        [(0.0, float(pv[t])) for t in range(T)]
    )

    res = linprog(c_obj, A_ub=np.array(A_rows), b_ub=np.array(b_rows),
                  bounds=bounds, method='highs')

    if res.status != 0:
        return np.zeros(T), np.zeros(T), pv.copy(), np.zeros(T)

    ch_grid = np.clip(res.x[:T],    0.0, p_max)
    di      = np.clip(res.x[T:2*T], 0.0, p_max)
    pv_sell = np.array([np.clip(res.x[2*T + t], 0.0, float(pv[t])) for t in range(T)])

    ch_grid[ch_grid < 0.1] = 0.0
    di[di < 0.1]           = 0.0
    pv_sell[pv_sell < 0.01] = 0.0
    pv_to_bat = np.maximum(pv - pv_sell, 0.0)

    for t in range(T):
        ch_tot = ch_grid[t] + pv_to_bat[t]
        if ch_tot > 0.0 and di[t] > 0.0:
            delta = ch_tot * eta_c - di[t] / eta_d
            if delta >= 0.0:
                scale        = min(delta / eta_c, p_max) / max(ch_tot, 1e-9)
                ch_grid[t]  *= scale
                pv_to_bat[t] = min(pv_to_bat[t] * scale, pv[t])
                pv_sell[t]   = pv[t] - pv_to_bat[t]
                di[t]        = 0.0
            else:
                ch_grid[t] = pv_to_bat[t] = 0.0
                pv_sell[t] = pv[t]
                di[t]      = min(-delta * eta_d, p_max)

    return ch_grid, di, pv_sell, pv_to_bat


# --- Test rapide -------------------------------------------------------------
test_px = np.array([20,15,10,12,30,60,80,50,25,20,
                    15,10,12,35,70,90,85,60,40,30,
                    20,15,12,10]) / 1000.0
test_pv = np.array([0,0,0,0,0,5,20,50,80,90,
                    80,60,40,20,10,5,0,0,0,0,
                    0,0,0,0], dtype=float)

for strat in [1, 2]:
    cg, di, pvs, pvb = optimize_schedule_pv(
        test_px, test_pv,
        soc_init_kwh = SOC_INIT_PCT * CAPACITY_KWH,
        capacity_kwh = CAPACITY_KWH, p_max = P_MAX,
        eta_c = ETA_C, eta_d = ETA_D,
        soc_min_pct  = SOC_MIN_PCT, soc_max_pct = SOC_MAX_PCT,
        agg_spread   = AGGREGATOR_SPREAD, fit_price = PV_FIT_PRICE,
        pv_strategy  = strat,
    )
    profit = (di * (test_px + AGGREGATOR_SPREAD) - cg * test_px + pvs * PV_FIT_PRICE).sum()
    print(f'Strat. {strat} — profit : {profit:.2f} EUR'
          f'  |  PV batt : {pvb.sum():.1f} kWh'
          f'  |  PV FIT  : {pvs.sum():.1f} kWh'
          f'  |  Ch. res : {cg.sum():.1f} kWh')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  SIMULATION HORIZON ROULANT AVEC PV
# ══════════════════════════════════════════════════════════════════════════════

def run_simulation_pv(df_sim, params, pv_array):
    """
    Simulation 2 re-plans/jour (00h et 13h) avec generateur PV.
    pv_array : ndarray, meme longueur que df_sim.
    """
    records        = []
    soc_kwh        = params['soc_init_pct'] * params['capacity_kwh']
    cap_kwh        = params['capacity_kwh']
    total_fec      = 0.0
    eta_c          = params['eta_c']
    eta_d          = params['eta_d']
    min_spread_kwh = params.get('min_discharge_spread_kwh', 0.0)
    fit_price      = params.get('fit_price', 0.10)
    pv_strategy    = params.get('pv_strategy', 2)
    last_charge_px = None

    dates  = sorted(set(df_sim.index.date))
    day_px = {d: df_sim[df_sim.index.date == d]['price_eur_kwh'].values for d in dates}
    pv_ser = pd.Series(pv_array, index=df_sim.index)
    day_pv = {d: pv_ser[pv_ser.index.date == d].values for d in dates}

    for day in dates:
        p_today  = day_px[day]
        pv_today = day_pv.get(day, np.zeros(24))
        if len(p_today) != 24:
            continue
        pv_today = np.pad(pv_today, (0, max(0, 24 - len(pv_today))))[:24]

        p_max   = min(params['c_rate'] * cap_kwh, params['connection_kw'])
        soc_min = params['soc_min_pct'] * cap_kwh
        soc_max = params['soc_max_pct'] * cap_kwh

        kw = dict(
            capacity_kwh=cap_kwh, p_max=p_max, eta_c=eta_c, eta_d=eta_d,
            soc_min_pct=params['soc_min_pct'], soc_max_pct=params['soc_max_pct'],
            agg_spread=params['agg_spread'], fit_price=fit_price,
            pv_strategy=pv_strategy,
        )

        ch_g_am, di_am, pvs_am, pvb_am = optimize_schedule_pv(
            p_today, pv_today, soc_kwh, **kw)

        ch_g_pm = di_pm = pvs_pm = pvb_pm = None
        ts_idx  = df_sim[df_sim.index.date == day].index

        for h in range(24):
            if h == 13:
                tomorrow = day + timedelta(days=1)
                if tomorrow in day_px and len(day_px[tomorrow]) == 24:
                    p_pm  = np.concatenate([p_today[13:],  day_px[tomorrow]])
                    pv_pm = np.concatenate([pv_today[13:], day_pv.get(tomorrow, np.zeros(24))])
                else:
                    p_pm  = p_today[13:]
                    pv_pm = pv_today[13:]
                ch_g_pm, di_pm, pvs_pm, pvb_pm = optimize_schedule_pv(
                    p_pm, pv_pm, soc_kwh, **kw)

            price = float(p_today[h])
            pv_h  = float(pv_today[h])

            if h < 13:
                ch_g_h = float(ch_g_am[h]);  di_h = float(di_am[h])
                pvs_h  = float(pvs_am[h]);   pvb_h = float(pvb_am[h])
            else:
                ch_g_h = float(ch_g_pm[h-13]); di_h = float(di_pm[h-13])
                pvs_h  = float(pvs_pm[h-13]);  pvb_h = float(pvb_pm[h-13])

            # Filtre spread minimum (charges reseau uniquement)
            if di_h > 0 and min_spread_kwh > 0 and last_charge_px is not None:
                if price - last_charge_px < min_spread_kwh:
                    di_h = 0.0

            # Plafonnement physique SOC
            ch_tot   = ch_g_h + pvb_h
            headroom = max(0.0, (soc_max - soc_kwh) / eta_c)
            if ch_tot > headroom:
                scale   = headroom / max(ch_tot, 1e-9)
                ch_g_h *= scale;  pvb_h *= scale;  pvs_h = pv_h - pvb_h
            di_h = min(di_h, max(0.0, (soc_kwh - soc_min) * eta_d))

            # Filet anti-simultaneite
            ch_tot = ch_g_h + pvb_h
            if ch_tot > 0.0 and di_h > 0.0:
                delta = ch_tot * eta_c - di_h / eta_d
                if delta >= 0.0:
                    scale   = min(delta / eta_c, p_max) / max(ch_tot, 1e-9)
                    ch_g_h *= scale;  pvb_h *= scale;  pvs_h = pv_h - pvb_h;  di_h = 0.0
                else:
                    ch_g_h = pvb_h = 0.0;  pvs_h = pv_h
                    di_h = min(-delta * eta_d, p_max)

            if ch_g_h > 0.01:
                last_charge_px = price

            soc_kwh += (ch_g_h + pvb_h) * eta_c - di_h / eta_d
            soc_kwh  = float(np.clip(soc_kwh, soc_min, soc_max))

            resale   = price + params['agg_spread']
            fec_inc  = (ch_g_h + pvb_h + di_h) / (2.0 * params['capacity_kwh'])
            total_fec += fec_inc

            records.append({
                'datetime':               ts_idx[h],
                'spot_price_eur_mwh':     round(price * 1000,            4),
                'pv_production_kwh':      round(pv_h,                    3),
                'pv_to_battery_kwh':      round(pvb_h,                   3),
                'pv_surplus_sold_kwh':    round(pvs_h,                   3),
                'charge_from_grid_kwh':   round(ch_g_h,                  3),
                'discharge_to_grid_kwh':  round(di_h,                    3),
                'net_flow_kwh':           round(ch_g_h + pvb_h - di_h,   3),
                'soc_kwh':                round(soc_kwh,                 3),
                'soc_pct':                round(soc_kwh / cap_kwh * 100, 2),
                'capacity_kwh':           round(cap_kwh,                 3),
                'purchase_cost_eur':      round(ch_g_h * price,          4),
                'pv_surplus_revenue_eur': round(pvs_h * fit_price,       4),
                'resale_revenue_eur':     round(di_h * resale,           4),
                'net_revenue_eur':        round(
                    di_h * resale - ch_g_h * price + pvs_h * fit_price,  4),
                'cumulative_fec':         round(total_fec,               4),
            })

        cap_kwh = params['capacity_kwh'] * max(
            params['capacity_eol'],
            1.0 - params['aging_per_fec'] * total_fec,
        )

    df_res = pd.DataFrame(records).set_index('datetime')

    n_simul = int(((df_res['charge_from_grid_kwh'] + df_res['pv_to_battery_kwh'] > 0) &
                   (df_res['discharge_to_grid_kwh'] > 0)).sum())
    print(f'  Verification : {n_simul} heures charge+decharge simultanees ({"OK" if n_simul==0 else "AVERT."})')
    return df_res


print('Fonctions definies OK.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PANNEAU PARAMETRES INTERACTIF
# ══════════════════════════════════════════════════════════════════════════════
if _HAS_WIDGETS:
    import time as _time

    _sty  = {'description_width': '230px'}
    _lw   = widgets.Layout(width='400px')
    _ls   = widgets.Layout(width='400px')

    # Batterie
    w_cap  = widgets.BoundedFloatText(value=CAPACITY_KWH, min=10, max=100_000, step=100,
                description='Capacite (kWh):', style=_sty, layout=_lw)
    w_cr   = widgets.FloatSlider(value=C_RATE, min=0.10, max=2.00, step=0.05,
                description='C-rate (h-1):', style=_sty, layout=_ls, readout_format='.2f')
    w_conn = widgets.BoundedFloatText(value=CONNECTION_KW, min=10, max=100_000, step=100,
                description='Raccordement (kW):', style=_sty, layout=_lw)
    w_eff  = widgets.FloatSlider(value=EFF_ROUNDTRIP, min=0.70, max=0.999, step=0.005,
                description='Rendement RT:', style=_sty, layout=_ls, readout_format='.3f')
    w_smin = widgets.FloatSlider(value=SOC_MIN_PCT, min=0.00, max=0.30, step=0.01,
                description='SOC minimum:', style=_sty, layout=_ls, readout_format='.0%')
    w_smax = widgets.FloatSlider(value=SOC_MAX_PCT, min=0.70, max=1.00, step=0.01,
                description='SOC maximum:', style=_sty, layout=_ls, readout_format='.0%')
    w_dsp  = widgets.FloatSlider(value=MIN_DISCHARGE_SPREAD_MWH, min=0, max=50, step=1,
                description='Spread min decharge (EUR/MWh):',
                style={'description_width':'230px'}, layout=_ls)

    # PV
    w_pvcap  = widgets.BoundedFloatText(value=PV_CAPACITY_KWP, min=1, max=10_000, step=10,
                  description='PV capacite (kWc):', style=_sty, layout=_lw)
    w_pvyld  = widgets.BoundedFloatText(value=PV_SPECIFIC_YIELD, min=500, max=2000, step=50,
                  description='Productible (kWh/kWc):', style=_sty, layout=_lw)
    w_pvfit  = widgets.FloatText(value=PV_FIT_PRICE,
                  description='Tarif FIT (EUR/kWh):', style=_sty, layout=_lw)
    w_pvlat  = widgets.FloatSlider(value=PV_LATITUDE, min=41.0, max=51.5, step=0.1,
                  description='Latitude (N):', style=_sty, layout=_ls, readout_format='.2f')
    w_pvlon  = widgets.FloatSlider(value=PV_LONGITUDE, min=-5.5, max=9.5, step=0.1,
                  description='Longitude (E):', style=_sty, layout=_ls, readout_format='.2f')
    w_pvstrat = widgets.Dropdown(
                  options=[
                      ('Strat. 2 — Priorite batterie (PV → batt. en premier)', 2),
                      ('Strat. 1 — Economique (arbitrage FIT / spot)',           1),
                  ],
                  value=PV_STRATEGY,
                  description='Strategie PV:',
                  style={'description_width': '120px'},
                  layout=widgets.Layout(width='500px'),
              )

    btn = widgets.Button(
        description='  LANCER LA SIMULATION',
        button_style='success', icon='play',
        layout=widgets.Layout(width='300px', height='44px', margin='10px 0 4px 0'),
    )
    out_widget = widgets.Output(
        layout=widgets.Layout(border='1px solid #b0c4de', border_radius='7px',
                               padding='10px 14px', margin_top='6px'))

    def _on_run(b):
        global CAPACITY_KWH, C_RATE, CONNECTION_KW, EFF_ROUNDTRIP
        global SOC_MIN_PCT, SOC_MAX_PCT, MIN_DISCHARGE_SPREAD_MWH
        global PV_CAPACITY_KWP, PV_SPECIFIC_YIELD, PV_FIT_PRICE, PV_STRATEGY
        global PV_LATITUDE, PV_LONGITUDE
        global ETA_C, ETA_D, P_MAX, MIN_DISCHARGE_SPREAD_KWH
        global pv_array, pv_profile, params, results

        CAPACITY_KWH             = w_cap.value
        C_RATE                   = w_cr.value
        CONNECTION_KW            = w_conn.value
        EFF_ROUNDTRIP            = w_eff.value
        SOC_MIN_PCT              = w_smin.value
        SOC_MAX_PCT              = w_smax.value
        MIN_DISCHARGE_SPREAD_MWH = w_dsp.value
        PV_CAPACITY_KWP          = w_pvcap.value
        PV_SPECIFIC_YIELD        = w_pvyld.value
        PV_FIT_PRICE             = w_pvfit.value
        PV_STRATEGY              = w_pvstrat.value
        PV_LATITUDE              = w_pvlat.value
        PV_LONGITUDE             = w_pvlon.value

        ETA_C = EFF_ROUNDTRIP ** 0.5
        ETA_D = EFF_ROUNDTRIP ** 0.5
        P_MAX = min(C_RATE * CAPACITY_KWH, CONNECTION_KW)
        MIN_DISCHARGE_SPREAD_KWH = MIN_DISCHARGE_SPREAD_MWH / 1000.0

        params = dict(
            capacity_kwh=CAPACITY_KWH, c_rate=C_RATE, connection_kw=CONNECTION_KW,
            eta_c=ETA_C, eta_d=ETA_D, soc_min_pct=SOC_MIN_PCT, soc_max_pct=SOC_MAX_PCT,
            soc_init_pct=SOC_INIT_PCT, agg_spread=AGGREGATOR_SPREAD,
            aging_per_fec=AGING_PER_FEC, capacity_eol=CAPACITY_EOL,
            min_discharge_spread_kwh=MIN_DISCHARGE_SPREAD_KWH,
            fit_price=PV_FIT_PRICE, pv_strategy=PV_STRATEGY,
        )

        b.description = '  Simulation en cours...'; b.disabled = True

        with out_widget:
            clear_output(wait=True)
            strat_names = {1: 'Economique', 2: 'Priorite batterie'}
            print(f'Strategie PV : {PV_STRATEGY} — {strat_names[PV_STRATEGY]}')
            print('Calcul profil PV...')
            pv_array   = compute_pv_profile(
                PV_CAPACITY_KWP, PV_SPECIFIC_YIELD,
                PV_LATITUDE, PV_LONGITUDE, df_sim.index)
            pv_profile = pd.Series(pv_array, index=df_sim.index)
            print(f'PV : {pv_array.sum():,.0f} kWh simules sur la periode')
            print()
            t0      = _time.time()
            results = run_simulation_pv(df_sim, params, pv_array)
            elapsed = _time.time() - t0

            n_days    = results.index.normalize().nunique()
            net_rev   = results['resale_revenue_eur'].sum()
            pv_rev    = results['pv_surplus_revenue_eur'].sum()
            net_cost  = results['purchase_cost_eur'].sum()
            net_prof  = results['net_revenue_eur'].sum()
            avg_d     = net_prof / n_days
            pv_tot    = results['pv_production_kwh'].sum()
            pv_bat    = results['pv_to_battery_kwh'].sum()

            print(f'Termine en {elapsed:.1f}s  |  {len(results):,} h  |  {n_days} jours')
            print('=' * 55)
            print(f'  Rev. decharge BESS     : {net_rev:>12,.2f} EUR')
            print(f'  Rev. surplus PV (FIT)  : {pv_rev:>12,.2f} EUR')
            print(f'  Cout achat reseau      : {net_cost:>12,.2f} EUR')
            print(f'  CONTRIBUTION NETTE     : {net_prof:>12,.2f} EUR')
            print(f'  Contribution / jour    : {avg_d:>12,.2f} EUR/jour')
            print(f'  Contribution annuelle  : {avg_d*365:>12,.0f} EUR/an')
            print(f'  Production PV simulee  : {pv_tot:>12,.0f} kWh')
            print(f'  Autoconsommation PV    : {pv_bat/max(pv_tot,1)*100:>11.1f} %')
            print('=' * 55)
            print()
            print('Relancer les cellules KPI / Graphiques / Export.')

        b.description = '  RELANCER LA SIMULATION'; b.disabled = False

    btn.on_click(_on_run)

    _col1 = widgets.VBox([w_cap, w_cr, w_conn, w_eff, w_smin, w_smax, w_dsp])
    _col2 = widgets.VBox([w_pvcap, w_pvyld, w_pvfit, w_pvlat, w_pvlon,
                          widgets.HTML('<b>Strategie de routage PV :</b>'),
                          w_pvstrat])
    _sep  = widgets.HTML('<div style="width:30px"></div>')

    _panel = widgets.VBox([
        widgets.HTML(
            '<h4 style="color:#1a3a5c;margin:0 0 10px 0">'
            'Parametres Batterie + PV — ajuster puis cliquer Lancer</h4>'),
        widgets.HBox([
            widgets.VBox([widgets.HTML('<b>Batterie</b>'), _col1]),
            _sep,
            widgets.VBox([widgets.HTML('<b>Generateur PV</b>'), _col2]),
        ]),
        widgets.HTML('<hr style="border-top:1px solid #d4e6f5;margin:10px 0">'),
        btn, out_widget,
    ])
    display(_panel)

In [ ]:
# ── Lancement par defaut (utiliser le panneau ci-dessus pour modifier) ────────
params = dict(
    capacity_kwh=CAPACITY_KWH, c_rate=C_RATE, connection_kw=CONNECTION_KW,
    eta_c=ETA_C, eta_d=ETA_D, soc_min_pct=SOC_MIN_PCT, soc_max_pct=SOC_MAX_PCT,
    soc_init_pct=SOC_INIT_PCT, agg_spread=AGGREGATOR_SPREAD,
    aging_per_fec=AGING_PER_FEC, capacity_eol=CAPACITY_EOL,
    min_discharge_spread_kwh=MIN_DISCHARGE_SPREAD_KWH,
    fit_price=PV_FIT_PRICE, pv_strategy=PV_STRATEGY,
)

_strat_label = {1: 'Economique (arbitrage FIT/spot)',
                2: 'Priorite batterie (PV → batt. en premier)'}
print(f'Strategie PV : {PV_STRATEGY} — {_strat_label[PV_STRATEGY]}')
print('Lancement de la simulation BESS + PV...')
t0      = time.time()
results = run_simulation_pv(df_sim, params, pv_array)
elapsed = time.time() - t0
print(f'Termine en {elapsed:.1f}s  =>  {len(results):,} enregistrements horaires')
print()
results.head(8)

In [ ]:
# ── KPI globaux ───────────────────────────────────────────────────────────────
total_rev    = results['resale_revenue_eur'].sum()
pv_rev       = results['pv_surplus_revenue_eur'].sum()
total_cost   = results['purchase_cost_eur'].sum()
net_profit   = results['net_revenue_eur'].sum()
n_days       = results.index.normalize().nunique()
avg_daily    = net_profit / n_days if n_days else 0

pv_total     = results['pv_production_kwh'].sum()
pv_to_bat    = results['pv_to_battery_kwh'].sum()
pv_sold      = results['pv_surplus_sold_kwh'].sum()
autoc_pct    = pv_to_bat / max(pv_total, 1) * 100

charged_kwh  = results['charge_from_grid_kwh'].sum()
disch_kwh    = results['discharge_to_grid_kwh'].sum()
fec_total    = results['cumulative_fec'].iloc[-1]
cap_final    = results['capacity_kwh'].iloc[-1]

print('=' * 62)
print('  RESULTATS — BESS + PV')
print('=' * 62)
print(f'  Periode            : {results.index[0].date()} => {results.index[-1].date()}')
print(f'  Duree              : {n_days} jours  ({len(results):,} heures)')
print()
print(f'  Rev. decharge BESS : {total_rev:>14,.2f} EUR')
print(f'  Rev. surplus PV    : {pv_rev:>14,.2f} EUR')
print(f'  Cout achat reseau  : {total_cost:>14,.2f} EUR')
print(f'  CONTRIBUTION NETTE : {net_profit:>14,.2f} EUR')
print(f'  Contrib. / jour    : {avg_daily:>14,.2f} EUR/jour')
print(f'  Contrib. annuelle  : {avg_daily*365:>13,.0f} EUR/an')
print()
print(f'  Production PV      : {pv_total:>14,.0f} kWh')
print(f'  PV vers batterie   : {pv_to_bat:>14,.0f} kWh  ({autoc_pct:.1f}%)')
print(f'  PV surplus vendu   : {pv_sold:>14,.0f} kWh')
print(f'  Achat reseau       : {charged_kwh:>14,.0f} kWh')
print(f'  Vente BESS         : {disch_kwh:>14,.0f} kWh')
print()
print(f'  FEC cumules        : {fec_total:>14.1f}')
print(f'  Capacite finale    : {cap_final:>14.1f} kWh  ({cap_final/CAPACITY_KWH*100:.2f}%)')
print('=' * 62)

# Bilan mensuel
monthly = results.resample('ME').agg(
    rev_bess = ('resale_revenue_eur',     'sum'),
    rev_pv   = ('pv_surplus_revenue_eur', 'sum'),
    cost     = ('purchase_cost_eur',      'sum'),
    profit   = ('net_revenue_eur',        'sum'),
    pv_prod  = ('pv_production_kwh',      'sum'),
    pv_bat_m = ('pv_to_battery_kwh',      'sum'),
    fec_c    = ('cumulative_fec',         'last'),
).round(2)
monthly.index = monthly.index.strftime('%Y-%m')
monthly['fec_mois'] = monthly['fec_c'].diff().fillna(monthly['fec_c'].iloc[0]).round(3)
monthly = monthly.drop(columns='fec_c')
print()
print('Bilan mensuel :')
print(monthly.to_string())

In [ ]:
# ── Graphique 1 : Premiere semaine — PV + flux batterie ───────────────────────
# Chercher une semaine d'ete pour voir le PV (par defaut premier lundi de juillet)
first_july = results[results.index.month == 7].index[0]
sample_start = first_july
sample = results.loc[sample_start:sample_start + pd.Timedelta(days=6, hours=23)]

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
fig.suptitle(f'Exploitation BESS + PV — semaine du {sample.index[0].date()}',
             fontsize=13, fontweight='bold')

# Prix spot
ax = axes[0]
ax.fill_between(sample.index, sample['spot_price_eur_mwh'], alpha=0.25, color='darkorange')
ax.plot(sample.index, sample['spot_price_eur_mwh'], color='darkorange', lw=1.5, label='Spot')
ax.axhline(0, color='black', lw=0.7, ls='--')
ax.set_ylabel('EUR/MWh');  ax.set_title('Prix spot France')
ax.legend(fontsize=9, loc='upper right')

# Production PV
ax = axes[1]
ax.fill_between(sample.index, sample['pv_production_kwh'], alpha=0.3, color='gold')
ax.plot(sample.index, sample['pv_production_kwh'], color='goldenrod', lw=1.5, label='Production PV')
ax.fill_between(sample.index, sample['pv_to_battery_kwh'], alpha=0.5, color='steelblue',
                label='PV vers batterie')
ax.fill_between(sample.index,
                sample['pv_to_battery_kwh'],
                sample['pv_to_battery_kwh'] + sample['pv_surplus_sold_kwh'],
                alpha=0.5, color='orange', label='PV surplus (FIT)')
ax.set_ylabel('kWh/h');  ax.set_title('Production PV et affectation')
ax.legend(fontsize=9, loc='upper right')

# Flux reseau
ax = axes[2]
w = 1.0/24.0 * 0.85
ax.bar(sample.index,  sample['charge_from_grid_kwh'],   color='royalblue',
       alpha=0.8, label='Charge reseau',  width=w)
ax.bar(sample.index, -sample['discharge_to_grid_kwh'],  color='tomato',
       alpha=0.8, label='Decharge BESS', width=w)
ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('kWh/h');  ax.set_title('Flux energie reseau')
ax.legend(fontsize=9, loc='upper right')

# SOC
ax = axes[3]
ax.fill_between(sample.index, sample['soc_pct'], alpha=0.2, color='steelblue')
ax.plot(sample.index, sample['soc_pct'], color='steelblue', lw=1.8, label='SOC')
ax.axhline(SOC_MIN_PCT*100, color='red',   ls='--', lw=1.0, label=f'SOC min {SOC_MIN_PCT*100:.0f}%')
ax.axhline(SOC_MAX_PCT*100, color='green', ls='--', lw=1.0, label=f'SOC max {SOC_MAX_PCT*100:.0f}%')
ax.set_ylabel('%');  ax.set_ylim(-2, 102);  ax.set_title('Etat de charge SOC')
ax.legend(fontsize=9, loc='upper right')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b %Hh'))
ax.xaxis.set_major_locator(mdates.HourLocator(byhour=[0, 6, 12, 18]))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=35, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('battery_PV_sample_week.png', dpi=130, bbox_inches='tight')
print('Sauvegarde : battery_PV_sample_week.png')
plt.show()

In [ ]:
# ── Graphique 2 : Vue d'ensemble — profit + PV mensuel ───────────────────────
daily = results.resample('D').agg(
    profit   = ('net_revenue_eur',       'sum'),
    pv_prod  = ('pv_production_kwh',     'sum'),
    pv_bat_d = ('pv_to_battery_kwh',     'sum'),
    capacity = ('capacity_kwh',          'last'),
    fec      = ('cumulative_fec',        'last'),
)

fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=True)
fig.suptitle('Vue ensemble — BESS + PV (periode complete)', fontsize=13, fontweight='bold')

# Profit journalier
ax = axes[0]
colors = ['steelblue' if v >= 0 else 'tomato' for v in daily['profit']]
ax.bar(daily.index, daily['profit'], color=colors, alpha=0.7, width=0.9)
ax.plot(daily.index, daily['profit'].rolling(30, min_periods=5).mean(),
        'k-', lw=2, label='Moy. mobile 30j')
ax.axhline(0, color='black', lw=0.6)
ax.set_ylabel('EUR/jour');  ax.set_title('Contribution nette journaliere (BESS + PV)')
ax.legend(fontsize=9)

# Production PV journaliere
ax = axes[1]
ax.fill_between(daily.index, daily['pv_bat_d'], alpha=0.5, color='gold', label='PV vers batterie')
ax.fill_between(daily.index, daily['pv_bat_d'], daily['pv_prod'],
                alpha=0.5, color='orange', label='PV surplus vendu')
ax.plot(daily.index, daily['pv_prod'].rolling(7).mean(), 'saddlebrown', lw=1.5, label='Moy. 7j PV total')
ax.set_ylabel('kWh/jour');  ax.set_title('Production PV journaliere (batterie + surplus)')
ax.legend(fontsize=9)

# Vieillissement
ax = axes[2]
ax.plot(daily.index, daily['capacity'], color='purple', lw=1.8, label='Capacite')
ax.axhline(CAPACITY_KWH, color='green', ls='--', lw=1,
           label=f'Nominale ({CAPACITY_KWH:.0f} kWh)')
ax.axhline(CAPACITY_KWH * CAPACITY_EOL, color='red', ls='--', lw=1,
           label=f'Fin de vie ({CAPACITY_EOL*100:.0f}%)')
ax2 = ax.twinx()
ax2.plot(daily.index, daily['fec'], color='gray', lw=1, alpha=0.5, ls='-.')
ax2.set_ylabel('FEC cumules', color='gray', fontsize=9)
ax2.tick_params(axis='y', labelcolor='gray')
ax.set_ylabel('kWh');  ax.set_title('Vieillissement batterie')
ax.legend(fontsize=9)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=35, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('battery_PV_full_period.png', dpi=130, bbox_inches='tight')
print('Sauvegarde : battery_PV_full_period.png')
plt.show()

In [ ]:
# ── Graphique 3 : Analyse PV — saisonnalite et autoconsommation ───────────────
monthly_pv = results.resample('ME').agg(
    pv_prod = ('pv_production_kwh',  'sum'),
    pv_bat  = ('pv_to_battery_kwh',  'sum'),
    pv_sell = ('pv_surplus_sold_kwh','sum'),
    rev_pv  = ('pv_surplus_revenue_eur','sum'),
    rev_bess= ('resale_revenue_eur', 'sum'),
    cost    = ('purchase_cost_eur',  'sum'),
    profit  = ('net_revenue_eur',    'sum'),
)
monthly_pv.index = monthly_pv.index.strftime('%Y-%m')
monthly_pv['autoc_pct'] = (monthly_pv['pv_bat'] /
                            monthly_pv['pv_prod'].replace(0, np.nan) * 100).fillna(0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Analyse PV — saisonnalite, affectation et contribution', fontsize=13)

# Production PV par mois
ax = axes[0]
x  = range(len(monthly_pv))
ax.bar(x, monthly_pv['pv_bat'],  bottom=0,                   color='gold',   alpha=0.85, label='PV batterie')
ax.bar(x, monthly_pv['pv_sell'], bottom=monthly_pv['pv_bat'],color='orange', alpha=0.80, label='PV surplus FIT')
ax.set_xticks(list(x))
ax.set_xticklabels(monthly_pv.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('kWh/mois');  ax.set_title('Production PV mensuelle')
ax.legend(fontsize=9)

# Taux d'autoconsommation
ax = axes[1]
colors_ac = ['steelblue' if v >= 60 else ('gold' if v >= 40 else 'tomato')
             for v in monthly_pv['autoc_pct']]
ax.bar(x, monthly_pv['autoc_pct'], color=colors_ac, alpha=0.8)
ax.axhline(autoc_pct, color='black', ls='--', lw=1.5,
           label=f'Moyenne {autoc_pct:.1f}%')
ax.set_xticks(list(x))
ax.set_xticklabels(monthly_pv.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('%');  ax.set_ylim(0, 105)
ax.set_title('Taux autoconsommation PV (%)')
ax.legend(fontsize=9)

# Contribution mensuelle decomposee
ax = axes[2]
ax.bar(x, monthly_pv['rev_bess'],   color='steelblue', alpha=0.85, label='Rev. BESS')
ax.bar(x, monthly_pv['rev_pv'],     bottom=monthly_pv['rev_bess'],
       color='gold', alpha=0.85, label='Rev. PV (FIT)')
ax.bar(x, -monthly_pv['cost'],      color='tomato', alpha=0.7, label='Cout achat')
ax.axhline(0, color='black', lw=0.6)
ax.set_xticks(list(x))
ax.set_xticklabels(monthly_pv.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('EUR/mois');  ax.set_title('Decomposition contribution mensuelle')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('battery_PV_analysis.png', dpi=130, bbox_inches='tight')
print('Sauvegarde : battery_PV_analysis.png')
plt.show()

In [ ]:
# ── Export CSV + JSON ──────────────────────────────────────────────────────────
export_cols = [
    'spot_price_eur_mwh',
    'pv_production_kwh', 'pv_to_battery_kwh', 'pv_surplus_sold_kwh',
    'charge_from_grid_kwh', 'discharge_to_grid_kwh', 'net_flow_kwh',
    'soc_pct', 'capacity_kwh',
    'purchase_cost_eur', 'pv_surplus_revenue_eur', 'resale_revenue_eur',
    'net_revenue_eur', 'cumulative_fec',
]
df_export = results[export_cols].copy()
df_export.index.name = 'datetime'
df_export.to_csv(OUTPUT_CSV_PATH, sep=';', decimal='.', encoding='utf-8-sig')
print(f'CSV exporte : {OUTPUT_CSV_PATH}  ({len(df_export):,} lignes)')

# Export JSON parametres
_params_json = {
    'battery': {
        'capacity_kwh': CAPACITY_KWH, 'c_rate': C_RATE,
        'connection_kw': CONNECTION_KW, 'p_max': P_MAX,
        'eff_roundtrip': EFF_ROUNDTRIP, 'aging_per_fec': AGING_PER_FEC,
        'capacity_eol': CAPACITY_EOL,
        'soc_min_pct': SOC_MIN_PCT, 'soc_max_pct': SOC_MAX_PCT,
        'agg_spread': AGGREGATOR_SPREAD,
        'min_discharge_spread_mwh': MIN_DISCHARGE_SPREAD_MWH,
    },
    'pv': {
        'capacity_kwp': PV_CAPACITY_KWP,
        'specific_yield_kwh_kwp': PV_SPECIFIC_YIELD,
        'annual_kwh': round(PV_CAPACITY_KWP * PV_SPECIFIC_YIELD),
        'fit_price_eur_kwh': PV_FIT_PRICE,
        'latitude': PV_LATITUDE,
        'longitude': PV_LONGITUDE,
    },
    'simulation': {
        'start': str(START_DATE),
        'end': str(results.index[-1].date()),
        'n_days': n_days,
    },
    'results_summary': {
        'net_profit_eur': round(net_profit, 2),
        'avg_daily_eur': round(avg_daily, 2),
        'annual_estimate_eur': round(avg_daily * 365),
        'pv_production_kwh': round(pv_total),
        'pv_autoconsommation_pct': round(autoc_pct, 1),
        'fec_total': round(fec_total, 1),
    },
}
with open('battery_PV_params.json', 'w', encoding='utf-8') as _f:
    json.dump(_params_json, _f, indent=2)
print('JSON exporte : battery_PV_params.json')
print()
print('Apercu des 5 premieres lignes :')
print(df_export.head(5).to_string())